[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/pml-f2026-notebooks/blob/main/class-demos/session10_logistic_regression.ipynb)

# Session 10 deck code, assembled in slide order

**Session 10 · the in-class demo — runs on the free tier of Colab, nothing to install**

Generated by scripts/make_demo.py from the slides themselves, so it stays
honest about what the deck actually shows. Run it before class.

Each heading names the slide its cell accompanies; the outputs below were saved from a real run.

## Slide 4: The Dataset: 200 Listings

In [1]:
import numpy as np

rng = np.random.default_rng(7)
n = 200
premium = rng.uniform(-10, 20, n)   # % over median
condition = rng.uniform(2, 10, n)   # 0-10 score

z = -1.2 - 0.30*premium + 0.45*condition
p = 1 / (1 + np.exp(-z))
sold_fast = (rng.random(n) < p).astype(int)

print(sold_fast[:12], sold_fast.sum(), "sold fast")

[1 0 0 1 1 0 1 0 0 0 1 1] 99 sold fast


## Slide 5: Attempt 1: Just Fit a Line

In [2]:
X = np.c_[np.ones(n), premium]
theta = np.linalg.lstsq(X, sold_fast, rcond=None)[0]
fit = X @ theta

print("theta", np.round(theta, 4))
print("range", np.round([fit.min(), fit.max()], 3))
bad = np.sum((fit < 0) | (fit > 1))
print("outside [0, 1]:", bad, "of", n)

theta [ 0.7071 -0.0417]
range [-0.121  1.119]
outside [0, 1]: 43 of 200


## Slide 17: From Scratch, Part 1: Features and Loss

In [3]:
X_raw = np.c_[premium, condition]
mu, sd = X_raw.mean(axis=0), X_raw.std(axis=0)
X_std = (X_raw - mu) / sd        # standardize
Xb = np.c_[np.ones(n), X_std]    # add bias column
y = sold_fast

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def log_loss(X, y, theta):
    p = sigmoid(X @ theta)
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -np.mean(y*np.log(p) + (1-y)*np.log(1-p))

## Slide 18: From Scratch, Part 2: Gradient (and Prove It)

In [4]:
def gradient(X, y, theta):
    m = len(y)
    return (1/m) * X.T @ (sigmoid(X @ theta) - y)

t, eps = np.array([0.3, -0.8, 0.5]), 1e-6
num = [(log_loss(Xb, y, t + eps*e)
        - log_loss(Xb, y, t - eps*e)) / (2*eps)
       for e in np.eye(3)]
print("analytic", np.round(gradient(Xb, y, t), 6))
print("numeric ", np.round(num, 6))

analytic [ 0.065256  0.191498 -0.002389]
numeric  [ 0.065256  0.191498 -0.002389]


## Slide 19: From Scratch, Part 3: The Training Loop

In [5]:
def fit_logistic(X, y, lr=0.5, epochs=4000):
    theta = np.zeros(X.shape[1])
    for ep in range(1, epochs + 1):
        theta -= lr * gradient(X, y, theta)
        if ep in {10, 100, 500, 4000}:
            L = log_loss(X, y, theta)
            print(f"ep {ep:>4}  loss {L:.4f}  "
                  f"theta {np.round(theta, 3)}")
    return theta

## Slide 20: From Scratch, Part 4: Run It

In [6]:
theta_gd = fit_logistic(Xb, y)
g = gradient(Xb, y, theta_gd)
print(f"||gradient|| = {np.linalg.norm(g):.2e}")

ep   10  loss 0.4025  theta [-0.013 -1.133  0.339]
ep  100  loss 0.3070  theta [ 0.011 -2.681  0.908]
ep  500  loss 0.3053  theta [ 0.019 -3.002  1.043]
ep 4000  loss 0.3053  theta [ 0.019 -3.004  1.043]
||gradient|| = 4.86e-16


## Slide 21: Moment of Truth: Us vs scikit-learn

In [7]:
from sklearn.linear_model import LogisticRegression

sk = LogisticRegression(penalty=None, max_iter=5000)
sk.fit(X_std, y)

print(np.round([sk.intercept_[0], *sk.coef_[0]], 4))
print(np.round(theta_gd, 4))   # ours

[ 0.0197 -3.0036  1.0431]
[ 0.0192 -3.0038  1.0432]


/Users/boyu/GitHub/PML_F2026/Python-for-Machine-Learning/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


## Slide 22: The C Trap: It Is Upside Down

In [8]:
for C in [100, 1, 0.1, 0.01]:
    m = LogisticRegression(C=C).fit(X_std, y)
    print(f"C={C:<6} w = {np.round(m.coef_[0], 3)}")

C=100    w = [-2.998  1.041]
C=1      w = [-2.594  0.88 ]
C=0.1    w = [-1.556  0.486]
C=0.01   w = [-0.492  0.146]


## Slide 23: Standardize Before You Regularize

In [9]:
for C in [100, 0.01]:
    r = LogisticRegression(C=C).fit(X_raw, y)
    s = LogisticRegression(C=C).fit(X_std, y)
    print(f"C={C:<5} raw {np.round(r.coef_[0], 3)}"
          f"  std {np.round(s.coef_[0], 3)}")

C=100   raw [-0.343  0.445]  std [-2.998  1.041]
C=0.01  raw [-0.269  0.205]  std [-0.492  0.146]


## Slide 24: Three Ways to Ask the Model

In [10]:
clf = LogisticRegression(C=1.0).fit(X_std, y)
P5 = clf.predict_proba(X_std[:5])
print(np.round(clf.decision_function(X_std[:5]), 3))
print(np.round(P5[:, 1], 3))
print(clf.predict(X_std[:5]), "predicted")
print(y[:5], "actual")

[-0.153 -3.033 -2.095  1.538  2.023]
[0.462 0.046 0.11  0.823 0.883]
[0 0 0 1 1] predicted
[1 0 0 1 1] actual


## Slide 25: The Threshold Is a Decision, Not a Model

In [11]:
proba = clf.predict_proba(X_std)[:, 1]
for th in [0.7, 0.5, 0.3]:
    flag = proba >= th
    hits = np.sum(flag & (y == 1))
    print(f"t={th}: flags {flag.sum():>3}, "
          f"catches {hits:>3} of {y.sum()}")

t=0.7: flags  85, catches  78 of 99
t=0.5: flags  97, catches  85 of 99
t=0.3: flags 119, catches  95 of 99


## Slide 27: Beyond Two Classes

In [12]:
S = np.c_[-1.2 - 0.30*premium + 0.45*condition,
          np.zeros(n),
          -2.2 + 0.26*premium - 0.30*condition]
P = np.exp(S - S.max(axis=1, keepdims=True))
P /= P.sum(axis=1, keepdims=True)

u = np.random.default_rng(21).random(n)[:, None]
outcome = (u > P.cumsum(axis=1)).sum(axis=1)
print(np.bincount(outcome), "fast/slow/gone")

[94 71 35] fast/slow/gone


## Slide 28: Strategy 1: One-vs-Rest

In [13]:
from sklearn.multiclass import OneVsRestClassifier

ovr = OneVsRestClassifier(LogisticRegression())
ovr.fit(X_std, outcome)

yes = [e.predict_proba(X_std[:1])[0, 1]
       for e in ovr.estimators_]
print(len(ovr.estimators_), "models fitted")
print("row 0", np.round(yes, 3))

3 models fitted
row 0 [0.407 0.35  0.077]


## Slide 29: Strategy 2: Softmax (Multinomial)

In [14]:
multi = LogisticRegression().fit(X_std, outcome)
Pm = multi.predict_proba(X_std)
print("coef_", multi.coef_.shape, "proba", Pm.shape)
print("row 0", np.round(Pm[0], 3))
print("sum", Pm[0].sum(), " acc",
      round(multi.score(X_std, outcome), 3))

coef_ (3, 2) proba (200, 3)
row 0 [0.421 0.504 0.075]
sum 1.0  acc 0.735


## Slide 31: When One Class Is Rare

In [15]:
rng_r = np.random.default_rng(33)
pr = sigmoid(-2.7 + 0.14*premium - 0.22*condition)
pulled = (rng_r.random(n) < pr).astype(int)
for cw in [None, "balanced"]:
    m = LogisticRegression(class_weight=cw)
    pred = m.fit(X_std, pulled).predict(X_std)
    hits = np.sum(pred & pulled)
    print(f"{str(cw):<8} flags {pred.sum():>3}, "
          f"catches {hits:>2} of {pulled.sum()}, "
          f"acc {m.score(X_std, pulled):.3f}")

None     flags   0, catches  0 of 15, acc 0.925
balanced flags  65, catches 13 of 15, acc 0.730
